# Fondamenti di Programmazione
TUTOR GIOVEDI 14-16 Aula Magna
## Laboratorio 03 — Test, debug, stringhe e liste

Oggi ripartiamo da un esercizio rimasto in sospeso nella scorsa esercitazione,
`cubic_root`, e lo useremo per introdurre un'abitudine fondamentale:

> **scrivere una funzione, provarla con dei test e correggerla se necessario.**

Poi passeremo agli esercizi del Laboratorio 3 su **stringhe e liste**, usando soltanto
strumenti che abbiamo già incontrato fino alla Lezione 4.

### Oggi useremo

- funzioni, parametri e `return`;
- `if`, `for`, `while`;
- stringhe, indici e slicing;
- liste;
- `append()`;
- `in` e `not in`;
- accumulatore;
- costruzione progressiva di un risultato.

Il metodo sarà sempre:

```text
capisco il problema
        ↓
penso a una strategia
        ↓
scrivo la funzione
        ↓
la testo
        ↓
se qualcosa non torna, faccio debug
```

# 1. Il file a celle in VS Code / Spyder

Nel file `.py` degli esercizi ogni blocco inizia con:

```python
# %%
```

In VS Code o Spyder possiamo eseguire una cella alla volta.

Questo è utile nel laboratorio perché possiamo:

1. definire una funzione;
2. eseguire solo quella cella;
3. eseguire subito i suoi test;
4. correggere il codice;
5. rieseguire.

La prima cella da eseguire è sempre:

```text
Funzioni per i test
```

# 2. Le funzioni per i test

Nel file troviamo due funzioni già pronte:

```python
print_test(...)
check_test(...)
```

Per ora le trattiamo come **scatole nere**.

Non è necessario capire ancora tutto il loro codice: contengono costrutti che
studieremo più avanti.

Ci interessa invece capire:

- che cosa ricevono;
- che cosa fanno;
- come leggere il risultato.

In [2]:
from typing import Any, Callable, List

# Queste funzioni servono per eseguire i test.
# Per ora NON è necessario capire come sono implementate.

def print_test(func: Callable, *args: List[Any]):
    func_str = func.__name__
    args_str = ", ".join(repr(arg) for arg in args)
    try:
        result = func(*args)
        result_str = repr(result)
        print(f"{func_str}({args_str}) => {result_str}")
    except BaseException as error:
        error_str = repr(error)
        print(f"[ERRORE] {func_str}({args_str}) => {error_str}")


def check_test(func: Callable, expected: Any, *args: List[Any]):
    func_str = func.__name__
    args_str = ", ".join(repr(arg) for arg in args)
    try:
        result = func(*args)
        result_str = repr(result)
        expected_str = repr(expected)

        if result == expected:
            print(
                f"[OK] {func_str}({args_str}) "
                f"=> {result_str}"
            )
        else:
            print(
                f"[FALLITO] {func_str}({args_str}) "
                f"=> {result_str} | atteso: {expected_str}"
            )
    except BaseException as error:
        error_str = repr(error)
        print(f"[ERRORE] {func_str}({args_str}) => {error_str}")

## `print_test()`

Consideriamo:

```python
print_test(cubic_root, -1)
```

Significa:

> esegui `cubic_root(-1)` e mostrami il risultato.

`print_test()` **non sa** quale dovrebbe essere il risultato corretto.

Serve quindi per osservare ciò che produce la funzione.

## `check_test()`

Consideriamo:

```python
check_test(cubic_root, 2.0, 8)
```

Leggiamolo da sinistra a destra:

```text
cubic_root  → funzione da testare

2.0         → risultato atteso

8           → argomento da passare alla funzione
```

Quindi il test esegue:

```python
cubic_root(8)
```

e verifica se il risultato è uguale a:

```python
2.0
```

## Tre risultati possibili

### `[OK]`

La funzione è stata eseguita e il risultato coincide con quello atteso.

### `[FALLITO]`

La funzione è stata eseguita, ma il risultato è diverso da quello atteso.

### `[ERRORE]`

Durante l'esecuzione Python ha generato un errore.

Questa distinzione è molto importante:

> **un test fallito non significa necessariamente che Python abbia dato un errore.**

Potremmo avere semplicemente scritto un algoritmo che calcola la cosa sbagliata.

In [3]:
def doppio(x):
    return x * 2

check_test(doppio, 10, 5)   # passa
check_test(doppio, 12, 5)   # fallisce

[OK] doppio(5) => 10
[FALLITO] doppio(5) => 10 | atteso: 12


# 3. Riprendiamo `cubic_root`

### Consegna

Scrivere una funzione:

```python
def cubic_root(n):
```

che restituisce la radice cubica di `n`.

Per esempio:

```python
cubic_root(8)
```

deve restituire:

```python
2.0
```

Prima di programmare chiediamoci:

1. qual è l'input?
2. qual è l'output?
3. quale operazione matematica dobbiamo fare?
4. quali casi conviene testare?

## Un primo tentativo

Supponiamo di scrivere:

```python
return n ** 1 / 3
```

Proviamo subito il test.

In [5]:
def cubic_root(n):
    return n ** 1 / 3

check_test(cubic_root, 2.0, 8)

[FALLITO] cubic_root(8) => 2.6666666666666665 | atteso: 2.0


Il programma viene eseguito, quindi **non abbiamo un errore di sintassi**.

Ma il test fallisce.

Perché?

Python interpreta:

```python
n ** 1 / 3
```

come:

```python
(n ** 1) / 3
```

Noi vogliamo invece elevare `n` all'esponente:

```python
1 / 3
```

Quindi dobbiamo scrivere:

```python
n ** (1 / 3)
```

Questo è un esempio di **errore logico**:

```text
il programma gira
ma calcola la cosa sbagliata
```

Il test ci aiuta ad accorgercene.

In [9]:
def cubic_root(n):
    return n ** (1 / 3)

check_test(cubic_root, 2.0, 8)
print(cubic_root(8))

2.0


## Testiamo anche un altro caso

Nel materiale troviamo anche:

```python
print_test(cubic_root, -1)
```

Proviamo.

In [10]:
print_test(cubic_root, -1)

cubic_root(-1) => (0.5000000000000001+0.8660254037844386j)


Il caso negativo ci costringe a ragionare.

Matematicamente:

```text
∛(-8) = -2
```

Possiamo quindi separare i casi:

```text
se n >= 0
    calcolo normalmente

altrimenti
    calcolo la radice di -n
    e rimetto il segno meno
```

In [14]:
def cubic_root(n):
    if n >= 0:
        return n ** (1 / 3)
    else:
        return -((-n) ** (1 / 3))


check_test(cubic_root, 2.0, 8)
check_test(cubic_root, -1.0, -1)
check_test(cubic_root, 0.0, 0)
check_test(cubic_root, -2.0, -8)

[OK] cubic_root(8) => 2.0
[OK] cubic_root(-1) => -1.0
[OK] cubic_root(0) => 0.0
[OK] cubic_root(-8) => -2.0


## Che cosa abbiamo fatto?

Non abbiamo scritto la funzione e basta.

Abbiamo seguito un ciclo di sviluppo:

```text
prima soluzione
      ↓
test
      ↓
test fallito
      ↓
capisco perché
      ↓
correggo
      ↓
aggiungo altri test
```

Questo è **debugging**.

Non significa soltanto "aggiustare errori di sintassi": significa capire perché il
programma non si comporta come ci aspettiamo.

# 4. Warm-up — `make_hello`

Passiamo ora agli esercizi del Laboratorio 3.

### Consegna

Scrivere una funzione che restituisce:

```text
Ciao <nome>. Buona giornata!
```

Esempio:

```python
make_hello("Pippo")
```

→

```text
"Ciao Pippo. Buona giornata!"
```

Qui riprendiamo:

- parametro;
- concatenazione di stringhe;
- `return`.

In [15]:
def make_hello(name):
    return "Ciao " + name + ". Buona giornata!"

#print_test riceve la funzione che voglio provare e poi gli argomenti da passarle.
#concettualemnte sto facendo make_hello("Pippo")
print_test(make_hello, "Pippo") #non mi dice se è corretto: mi mostra quello che la funzione ha restituito.


check_test(
    make_hello, #funzione da testare
    "Ciao Pippo. Buona giornata!", #risultato atteso
    "Pippo" #argomento che gli passo
)
#output: [OK] make_hello('Pippo') => 'Ciao Pippo. Buona giornata!' mi dice appunto se corretta: OK

#Esempio fallimento:

check_test(
    make_hello,
    "Ciao Pippo!",
    "Pippo"
)

#La funzione non ha nessun errore. Semplicemente restituisce: "Ciao Pippo. Buona giornata!"
# mentre noi abbiamo dichiarato che ci aspettavamo: "Ciao Pippo!"
# [FALLITO]


make_hello('Pippo') => 'Ciao Pippo. Buona giornata!'
[OK] make_hello('Pippo') => 'Ciao Pippo. Buona giornata!'
[FALLITO] make_hello('Pippo') => 'Ciao Pippo. Buona giornata!' | atteso: 'Ciao Pippo!'


# 5. `strip_whitespace`: tre soluzioni

Rimuovere gli spazi all’inizio e alla fine senza usare `str.strip()`. Vediamo prima una soluzione esplicita e poi due versioni più compatte.

## Versione 1 — `while` + `if` + `break`

Usiamo `start` ed `end`. Finché siamo dentro la stringa, se troviamo uno spazio avanziamo; al primo carattere utile usciamo con `break`.

In [ ]:
def strip_whitespace_v1(string):
    start = 0
    end = len(string) - 1

    while start < len(string):
        if string[start] == " ":
            start += 1
        else:
            break

    while end >= 0:
        if string[end] == " ":
            end -= 1
        else:
            break

    return string[start:end + 1]

check_test(strip_whitespace_v1, "Pippo", "  Pippo  ")
check_test(strip_whitespace_v1, "", "   ")
check_test(strip_whitespace_v1, "Pippo", "Pippo")

## Tracing

Con `string = "   Harry  "`, `start` avanza finché incontra `H`; `end` arretra finché incontra `y`. Alla fine usiamo `string[start:end + 1]`: nello slicing l’estremo finale è escluso.

# Versione 2 — la condizione direttamente nel `while`

Il ciclo deve continuare solo se l’indice è valido **e** il carattere è uno spazio. Possiamo incorporare l’`if` nella condizione del `while`.

In [ ]:
def strip_whitespace_v2(string):
    start = 0
    end = len(string) - 1

    while start < len(string) and string[start] == " ":
        start += 1

    while end >= 0 and string[end] == " ":
        end -= 1

    return string[start:end + 1]

check_test(strip_whitespace_v2, "Pippo", "  Pippo  ")
check_test(strip_whitespace_v2, "", "   ")
check_test(strip_whitespace_v2, "Pippo", "Pippo")

## Short-circuit

Con una stringa di soli spazi, quando `start == len(string)` la prima condizione è `False`. Python non valuta `string[start]`, evitando `IndexError`. Per questo l’ordine delle condizioni è importante.

# Versione 3 — senza indici espliciti

Possiamo eliminare progressivamente gli spazi usando lo slicing. Le stringhe sono immutabili: `string = string[1:]` crea una nuova stringa e riassegna il nome.

In [ ]:
def strip_whitespace_v3(string):
    while string and string[0] == " ":
        string = string[1:]

    while string and string[-1] == " ":
        string = string[:-1]

    return string

check_test(strip_whitespace_v3, "Pippo", "  Pippo  ")
check_test(strip_whitespace_v3, "", "   ")
check_test(strip_whitespace_v3, "Pippo", "Pippo")

## `while string`

La stringa vuota `""` vale `False` in una condizione. Quindi `string and string[0] == " "` controlla prima che la stringa non sia vuota; grazie allo short-circuit non accede a `string[0]` quando non esiste.

# Confronto

- **V1:** più esplicita; utile per capire il ragionamento.
- **V2:** stessa logica, ma la condizione è direttamente nel `while`.
- **V3:** usa slicing, truth value delle stringhe e short-circuit.

> **Prima correttezza e comprensione, poi compattezza ed eleganza.**

# 6. Liste — `sum_squares`

### Consegna

Scrivere una funzione che restituisce la somma dei quadrati degli elementi.

```python
sum_squares([1, 2, 3])
```

→

```text
1² + 2² + 3² = 14
```

Qui riutilizziamo un pattern fondamentale:

## L'accumulatore

```python
totale = 0

for elemento in ...:
    totale += ...
```

In [ ]:
def sum_squares(elements):
    totale = 0

    for element in elements:
        totale += element * element

    return totale


check_test(sum_squares, 14, [1, 2, 3])
check_test(sum_squares, 20, [2, 4])
check_test(sum_squares, 0, [])

# 7. Liste — `max_element`

### Consegna

Scrivere una funzione che restituisce il massimo elemento della lista senza usare:

```python
max()
```

### Quale informazione dobbiamo ricordare?

Il **massimo trovato fino a questo momento**.

Attenzione a non inizializzare:

```python
massimo = 0
```

perché una lista potrebbe contenere solo numeri negativi:

```python
[-1, -2]
```

In [ ]:
def max_element(elements):
    if len(elements) == 0:
        return None

    massimo = elements[0]

    for element in elements:
        if element > massimo:
            massimo = element

    return massimo


check_test(max_element, 3, [1, 2, 3, -1, -2])
check_test(max_element, -1, [-1, -2])
check_test(max_element, None, [])

# 8. Liste — `remove_duplicates`

### Consegna

Scrivere una funzione che restituisce una nuova lista senza duplicati.

```python
[1, 2, 3, 2, 3]
```

→

```python
[1, 2, 3]
```

Vogliamo mantenere l'ordine della **prima comparsa**.

### Strategia

Costruiamo una nuova lista:

```python
result = []
```

Per ogni elemento chiediamo:

```text
è già presente in result?
```

Se non è presente, lo aggiungiamo.

In [ ]:
def remove_duplicates(elements):
    result = []

    for element in elements:
        if element not in result:
            result.append(element)

    return result


check_test(
    remove_duplicates,
    [1, 2, 3],
    [1, 2, 3, 2, 3]
)

check_test(
    remove_duplicates,
    ["Harry", "Ron", "Luna"],
    ["Harry", "Ron", "Harry", "Luna"]
)

## Perché non usare semplicemente un set?

Abbiamo appena studiato:

```python
set
```

e sappiamo che elimina i duplicati.

Ma qui vogliamo anche mantenere l'ordine della prima comparsa.

Inoltre la soluzione con una lista ci permette di allenare:

- `for`;
- `if`;
- `not in`;
- costruzione progressiva di un contenitore.

# 9. Liste — `reverse_list`

### Consegna

Scrivere una funzione che restituisce una **nuova lista** con gli elementi in ordine inverso.

Non usare:

```python
reverse()
reversed()
```

Esempio:

```python
[1, 2, 3]
```

→

```python
[3, 2, 1]
```

## Strategia

Partiamo da:

```python
result = []
```

Per ogni elemento costruiamo una nuova lista mettendo il nuovo elemento **davanti**:

```python
result = [element] + result
```

Seguiamo a mano:

```text
inizio       []

leggo 1      [1]

leggo 2      [2, 1]

leggo 3      [3, 2, 1]
```

In [ ]:
def reverse_list(elements):
    result = []

    for element in elements:
        result = [element] + result

    return result


check_test(reverse_list, [3, 2, 1], [1, 2, 3])

# 10. Challenge — `split_string`

Se abbiamo completato gli esercizi precedenti, proviamo un esercizio più articolato.

### Consegna

Scrivere una funzione che divide una stringa quando incontra uno dei caratteri
contenuti in `characters`.

Per esempio:

```python
split_string("Pippo   Pluto", " ")
```

→

```python
["Pippo", "Pluto"]
```

Non usare `str.split()`.

Non vogliamo stringhe vuote nel risultato.

## Strategia

Manteniamo due variabili:

```python
result = []
current = ""
```

`current` contiene la parola che stiamo costruendo.

Per ogni carattere:

```text
se è un separatore:
    se current non è vuota:
        aggiungila a result
        ricomincia con current = ""

altrimenti:
    aggiungi il carattere a current
```

Alla fine dobbiamo ricordarci di aggiungere l'ultima parola.

In [ ]:
def split_string(string, characters):
    result = []
    current = ""

    for character in string:
        if character in characters:
            if current != "":
                result.append(current)
                current = ""
        else:
            current += character

    if current != "":
        result.append(current)

    return result


check_test(
    split_string,
    ["Pippo", "Pluto"],
    "Pippo Pluto",
    " "
)

check_test(
    split_string,
    ["Pippo", "Pluto"],
    "Pippo   Pluto",
    " "
)

check_test(
    split_string,
    ["uno", "due", "tre"],
    "uno,due;tre",
    ",;"
)

# 11. Riepilogo del laboratorio

Oggi abbiamo allenato soprattutto il **metodo**.

```text
CONSEGNA
   ↓
input / output
   ↓
strategia
   ↓
implementazione
   ↓
test
   ↓
debug
```

Abbiamo incontrato diversi tipi di problema:

- errore logico (`cubic_root`);
- ricerca di indici (`strip_whitespace`);
- accumulatore (`sum_squares`);
- miglior valore trovato finora (`max_element`);
- costruzione progressiva di una lista (`remove_duplicates`);
- tracing di un ciclo (`reverse_list`);
- composizione di stringhe e liste (`split_string`).

Quando un programma non funziona, non cambiamo codice a caso:

> **scegliamo un caso semplice, seguiamo le variabili passo passo e confrontiamo ciò che accade con ciò che ci aspettavamo.**